In [2]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob
from matplotlib.colors import LogNorm
import pandas as pd
from pathlib import Path

from collections import defaultdict


In [4]:
# get int run files in experiment path
def get_runs(experiment_path):
    runs = !ls $experiment_path
    runs = np.sort(np.asarray(runs).astype(int))
    return runs   

# get paths for each snap file for each folder in experiment_path
def get_snaps_many(experiment_path):
    runs = get_runs(experiment_path)

    snaps = {}
    for run in runs:
        #snaps[run] = !ls $experiment_path/$run/snap.40_*.h5part
        pattern = (Path(experiment_path) / str(run)).glob("snap.40_*.h5part")
        snaps[run] = sorted(pattern)

    return runs, snaps

# get paths for snapshots in a folder
def get_snaps(path):
    #snaps = !ls $path/snap.40_*.h5part
     return sorted(Path(path).glob("snap.40_*.h5part"))

# convert global.30 in run_path to a dataframe
def get_glob_df(run_path):
    glob_path = Path(run_path) / 'global.30'
    return pd.read_csv(glob_path, sep=r"\s+", index_col=False)

# find conversion between nb time and myr using the path to a glabal.30 file
def get_Myr_per_Nbody(run_path):
    #https://stackoverflow.com/questions/46258499/how-to-read-the-last-line-of-a-file-in-python
    glob_path = Path(run_path) / 'global.30'
    with open(glob_path, 'rb') as f:
        try:  
            f.seek(-2, os.SEEK_END)
            while f.read(1) != b'\n':
                f.seek(-2, os.SEEK_CUR)
        except OSError:
            f.seek(0)
        nb, myr = f.readline().decode().split()[:2]
        return float(myr) / float(nb)
    
# build a df storing time and path data about a snapshot
def get_snap_times(run, snap_path, Myr_per_NB):
    with h5py.File(snap_path, 'r') as f:
        
        df = pd.DataFrame()
        df['Step'] = sorted(f.keys(), key=lambda k: f[k].attrs['Time'])
        df['Time[NB]'] = [f[step].attrs['Time'] for step in df['Step']]
        df['Time[Myr]'] = df['Time[NB]'] * Myr_per_NB
        df['snap_path'] = snap_path
        df["snap_name"] = Path(snap_path).name
        df['run'] = run
        return df
        
# build a df storing time and path data about multipl snapshots
def get_run_snap_times(run, snaps, glob_df):
    Myr_per_NB = glob_df['TIME[Myr]'].iloc[-1] / glob_df['TIME[NB}'].iloc[-1]

    dfs = []
    for snap_path in snaps:
        dfs.append(get_snap_times(run, snap_path, Myr_per_NB))

    return pd.concat(dfs, ignore_index=True).sort_values("Time[Myr]")

# build a df storing time and path data about every snapshot in an experiment
def get_experiment_snap_times(experiment_path):
    runs, snaps = get_snaps_many(experiment_path)

    dfs = []
    for run in runs:
        run_path = Path(f'{experiment_path}/{run}')
        glob_df = get_glob_df(run_path)

        dfs.append(get_run_snap_times(run, snaps[run], glob_df))

    return pd.concat(dfs, ignore_index=True).sort_values("Time[Myr]")
        
# find the step and snapshot thats closest to a list of times
def find_times_Myr(snap_df, run, times):
    mask = snap_df['run'] == run

    idx = np.searchsorted(snap_df.loc[mask, 'Time[Myr]'], times)    
    idx[idx >= len(snap_df[mask])] = len(snap_df[mask]) - 1
    
    return snap_df[mask].iloc[idx].reset_index(drop=True)

# find roughly how many steps in snap files to skip to take a reading every myr_interval Myr
def find_skip(run_path, myr_interval, step_interval=0.125):
    Myr_per_Nbody = get_Myr_per_Nbody(run_path)

    nb_interval = myr_interval / Myr_per_Nbody
    step_skip = nb_interval / step_interval
    return max(1, int(step_skip))

# build a df for a step in a snap file 
def get_step(snap_path, step, Myr_per_Nbody):
    with h5py.File(snap_path, 'r') as f:
        s = f[step]
        df = pd.DataFrame({
                    "M": s["M"][:],
                    "NAM": s["NAM"][:],
                    "POT": s["POT"][:],
                    "Vx": s["V1"][:], "Vy": s["V2"][:], "Vz": s["V3"][:],
                    "X":  s["X1"][:], "Y":  s["X2"][:], "Z":  s["X3"][:]})
        df.attrs['Time'] = s.attrs["Time"] * Myr_per_Nbody

        return df
        

In [6]:
(0.125 * 10.2) // 0.125

10.0

In [10]:
# get snapshot data for every 'step_interval' steps
def get_steps(path, snap_interval=20, step_skip=1):
    Myr_per_Nbody = get_Myr_per_Nbody(path)
    data = {}
    for snap in get_snaps(path):
        with h5py.File(snap, 'r') as f:
            for step in f.keys():
                step_num = int(step.split("#")[1])
                if step_num % step_skip != 0: # skip non multiples of step_interval
                    continue

                s = f[step]
                df = pd.DataFrame({
                    "M": s["M"][:],
                    "NAM": s["NAM"][:],
                    "POT": s["POT"][:],
                    "Vx": s["V1"][:], "Vy": s["V2"][:], "Vz": s["V3"][:],
                    "X":  s["X1"][:], "Y":  s["X2"][:], "Z":  s["X3"][:]})
                time = s.attrs["Time"] * Myr_per_Nbody
                df.attrs['Time'] = time
                data[time] = df

   
    return dict(sorted(data.items())), Myr_per_Nbody
        
    
# get closest snapshot data (and global.30, Myr_per_Nbody) for a set of times / Myr
def get_steps_at_times(path, times, run='', snap_interval=20, step_interval=0.125):
    Myr_per_Nbody = get_Myr_per_Nbody(path)

    # find the snap and step within that each time would be stored at 
    times = np.asarray(times)
    nbody_times = times / Myr_per_Nbody
    snap_at_time = (nbody_times // snap_interval * snap_interval).astype(int)
    step_at_snap = ((nbody_times - snap_at_time) // step_interval).astype(int)

    snaps = get_snaps(path)
    requested_snaps = defaultdict(list)
    for target_time, snap_num, step_num in zip(times, snap_at_time, step_at_snap):
        request = Path(path) / f'snap.40_{snap_num}.h5part'
        if request not in snaps:
            print(f'{target_time}Myr not found in run {run}')
            continue

        requested_snaps[request].append((target_time, f'Step#{step_num}'))


    data = {}
    for snap, step_requests in requested_snaps.items():
        with h5py.File(snap, 'r') as f:      
            for target_time, step in step_requests:
                s = f[step]
                
                df = pd.DataFrame({
                    "M": s["M"][:],
                    "NAM": s["NAM"][:],
                    "POT": s["POT"][:],
                    "Vx": s["V1"][:],
                    "Vy": s["V2"][:],
                    "Vz": s["V3"][:],
                    "X": s["X1"][:],
                    "Y": s["X2"][:],
                    "Z": s["X3"][:],
                })
                df.attrs['Time'] = s.attrs["Time"] * Myr_per_Nbody

                data[target_time] = df
                
    return data, Myr_per_Nbody 


#get data about a choice of runs (default to all), 
#pass a list of times to 'times' to only read at the corrisponding steps
#to read a value every n Myr, pass float n to Myr_skip
#    or to read a value every n steps, pass int n to step_skip
def get_runs_data(experiment_path, times=None, runs=None, step_skip=1, Myr_skip=None):
    if runs is None:
        runs = get_runs(experiment_path)
        
    run_paths = {run:Path(experiment_path) / str(run) for run in runs}
    if times is not None:
        runs_data = {run : get_steps_at_times(path, times, run) for run, path in run_paths.items()}
    else:
        if Myr_skip is not None:
            step_skips = {run: find_skip(path, Myr_skip) for run, path in run_paths.items()}
            runs_data = {run : get_steps(path, step_skip=step_skips[run]) for run, path in run_paths.items()}
        else:
            runs_data = {run : get_steps(path, step_skip=step_skip) for run, path in run_paths.items()}
        
    return runs_data

In [ ]:
# get closest snapshot data (and global.30, Myr_per_Nbody) for a set of times / Myr
def get_steps_at_times_old(path, times, run=''):
    glob_df = get_glob_df(path)
    Myr_per_Nbody = glob_df['TIME[Myr]'].iloc[-1] / glob_df['TIME[NB}'].iloc[-1]

    times = np.asarray(times)
    nbody_times = times / Myr_per_Nbody
    snap_at_time = (nbody_times // snap_interval * snap_interval).astype(int)

    snaps = get_snaps(path)
    requested_snaps = defaultdict(list)
    for time, snap_num in zip(times, snap_at_time):
        request = Path(path) / f'snap.40_{snap_num}.h5part'
        if request not in snaps:
            print(f'{time}Myr not found in run {run}')
            continue

        requested_snaps[request].append(time)

    
    data = {}
    for snap, times in requested_snaps.items():
        with h5py.File(snap, 'r') as f:
            step_times = {step: f[step].attrs['Time'] for step in f.keys()}

            request_times = requested_snaps[snap]

            
            for time in times:
                ntime = time / Myr_per_Nbody
                closest = min(step_times, key=lambda s: abs(step_times[s] - ntime))
                s = f[closest]
                
                df = pd.DataFrame({
                    "M": s["M"][:],
                    "NAM": s["NAM"][:],
                    "POT": s["POT"][:],
                    "Vx": s["V1"][:],
                    "Vy": s["V2"][:],
                    "Vz": s["V3"][:],
                    "X": s["X1"][:],
                    "Y": s["X2"][:],
                    "Z": s["X3"][:],
                })
                df.attrs['Time'] = s.attrs["Time"] * Myr_per_Nbody

                data[time] = df
                
    return data, glob_df, Myr_per_Nbody 

In [ ]:
# dont use this 
# plot a step in a snap file on the given ax 
def plot_step(fig, ax, snap_path, step, Myr_per_Nbody, offset=200, label=None,
              hidexlbl=False, hideylbl=False, hidetitle=False):   
    
    stepdf = get_step(snap_path, step, Myr_per_Nbody)
    #print(stepdf)
        
    #com_x = np.average(x, weights=m)
    #com_z = np.average(z, weights=m)

    com_x = np.median(stepdf["X"])
    com_y = np.median( stepdf["Y"])

    hist, xedges, yedges = np.histogram2d(stepdf['X'] - com_x, stepdf['Y'] - com_y, bins=100,
                                          range=[[-offset, offset], [-offset, offset]])
    ax.imshow(
    hist.T,
    origin='lower',
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
    cmap='gray_r',
    norm=LogNorm(),
    label=label
    )
    
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
    
    if not hidetitle: ax.set_title(f'{stepdf.attrs['Time']:.1f} Myr')
    if not hidexlbl: ax.set_xlabel('X / pc')
    if not hideylbl: ax.set_ylabel('Y / pc')
    if label is not None: ax.text(-0.8*offset, 0.8*offset, label)
    ax.set_xlim(-offset, offset)
    ax.set_ylim(- offset, offset)
    ax.set_aspect('equal')    

In [27]:
def plot_clustergram(fig, ax, xdata, ydata, offset=200, label='',
              xlbl='', ylbl='', title='', bins=40): 
    com_x = np.median(xdata)
    com_y = np.median(ydata)

    hist, xedges, yedges = np.histogram2d(xdata - com_x, ydata - com_y, bins=bins,
                                          range=[[-offset, offset], [-offset, offset]])

    
    ax.imshow(
    np.log10(hist.T + 1),
    origin='lower',
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
    cmap='gray_r',
    #norm=LogNorm(),
    label=label,
    interpolation='spline36'
    )

    ax.set_title(title)
    ax.set_xlabel(xlbl)
    ax.set_ylabel(ylbl)
    ax.text(-0.8*offset, 0.8*offset, label)
    ax.set_xlim(-offset, offset)
    ax.set_ylim(- offset, offset)
    ax.set_aspect('equal')    
    
    
def plot_clustergram_rad(fig, ax, latdata, londata, m=None, lat_offset=None, lon_offset=None, label='',
              xlbl='', ylbl='', title='', bins=40, acceptance=95):

    latdata = np.array(latdata, dtype=float)
    londata = np.array(londata, dtype=float)

    if lat_offset is None:
        lat_offset = np.percentile(np.abs(latdata - np.median(latdata)), acceptance)
    if lon_offset is None:
        lon_offset = np.percentile(np.abs(londata - np.median(londata)), acceptance)

    com_lat = np.median(latdata)
    com_lon = np.median(londata)

    hist, xedges, yedges = np.histogram2d(
        londata - com_lon, latdata - com_lat,
        bins=bins,
        range=[[-lon_offset, lon_offset], [-lat_offset, lat_offset]],
        weights=m
    )

    ax.imshow(
        np.log10(hist.T + 1),
        origin='lower',
        extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
        cmap='gray_r',
        label=label,
        interpolation='spline36'
    )
    ax.set_title(title)
    ax.set_xlabel(xlbl)
    ax.set_ylabel(ylbl)
    ax.text(-0.8 * lon_offset, 0.8 * lat_offset, label)